# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [1]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append("/home/shikim/pynta")
#sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))
from ase import Atom, Atoms
import copy
from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface,fcc111
from ase.visualize import view
from site_analysis import *

In [2]:

# Load pre-made slab
slab=read('/home/shikim/pynta/pynta/preprocessing/custom_surfaces/testing_shinae/defects/pt111_defect_middle.xyz')

nslab = len(slab)
adsorbate_height = 2    
site_bond_cutoff = 1.5
view(slab,viewer='x3d')
pt111_defect = CustomSurface(slab, n_layers=4 )

In [3]:
cas = SlabAdsorptionSites(slab, surface=pt111_defect, composition_effect=True)  # ACAT has surface, custom does not find them all!
single_geoms, single_sites_lists = generate_unique_sites(
        slab,
        cas.get_sites(),
        nslab,
        site_bond_cutoff,
        adsorbate_height
    )

In [4]:

# Extract site graphs
admols, geom_indices = classify_all_sites(
    single_geoms, single_sites_lists
)


In [5]:

# Graph isomorphism clustering
iso_mat, clusters = cluster_isomorphic_graphs(admols)


In [6]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
0 vs 3 = False
0 vs 4 = False
0 vs 5 = False
0 vs 6 = False
0 vs 7 = False
0 vs 8 = False
0 vs 9 = False
0 vs 10 = False
0 vs 11 = False
0 vs 12 = False
0 vs 13 = False
0 vs 14 = False
0 vs 15 = False
0 vs 16 = False
0 vs 17 = False
0 vs 18 = False
1 vs 2 = True
1 vs 3 = True
1 vs 4 = True
1 vs 5 = False
1 vs 6 = True
1 vs 7 = True
1 vs 8 = True
1 vs 9 = True
1 vs 10 = True
1 vs 11 = True
1 vs 12 = True
1 vs 13 = True
1 vs 14 = True
1 vs 15 = False
1 vs 16 = True
1 vs 17 = False
1 vs 18 = False
2 vs 3 = True
2 vs 4 = True
2 vs 5 = False
2 vs 6 = True
2 vs 7 = True
2 vs 8 = True
2 vs 9 = True
2 vs 10 = True
2 vs 11 = True
2 vs 12 = True
2 vs 13 = True
2 vs 14 = True
2 vs 15 = False
2 vs 16 = True
2 vs 17 = False
2 vs 18 = False
3 vs 4 = True
3 vs 5 = False
3 vs 6 = True
3 vs 7 = True
3 vs 8 = True
3 vs 9 = True
3 vs 10 = True
3 vs 11 = True
3 vs 12 = True
3 vs 13 = True
3 vs 14 = True
3 vs 15 = False
3 vs 16 = True
3 vs 17 = Fal

In [7]:

# Assuming single_sites_lists and clusters are defined# Assuming single_sites_lists and clusters are defined
updated_sites, site_mapping = update_site_labels(single_sites_lists, clusters)

Clusters structure: {0: [0], 1: [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 16], 5: [5], 15: [15], 17: [17], 18: [18]}
Single sites lists structure: [[{'site': 'ontop', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([12.47320201,  7.20148072, 19.00418576]), 'normal': array([-3.534e-05,  8.600e-07,  1.000e+00]), 'indices': (27,), 'composition': 'Pt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27])}], [{'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([ 5.54376719,  7.20142653, 19.00491345]), 'normal': array([-1.3232000e-04,  4.2990000e-05,  9.9999999e-01]), 'indices': (27, 28), 'composition': 'PtPt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28])}], [{'site': '3fold', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([ 1.38594787,  0.80021856, 19.0051173 ]), 'normal': array([-5.2499000e-04, -2.5474000e-04,  9.9999983e-01

In [8]:
write_trajectory_pynta(slab, cas, nslab, site_bond_cutoff, adsorbate_height)

There are 19 unique sites out of 45.
Sites data saved to 'sites.json' with None replaced by 'null'.


In [9]:
save_neighbor_site_list_to_json(cas)

Neighbor site list saved to 'neighbor_site_list.json'.


# defects

In [10]:
xyz_path = '/home/shikim/pynta/pynta/preprocessing/custom_surfaces/testing_shinae/defects/pt111_defect_middle.xyz'
slab = read(xyz_path)
nslab = len(xyz_path)
view(slab,viewer='x3d')


In [11]:
defect_sites, _ = detect_vacancy_sites_from_coordination(slab, nslab)

In [12]:
print(f"Found {len(defect_sites)} defect site(s)")
for s in defect_sites:
    x, y, z = s["position"]
    print(f"{s.get('name','defect')}: x={x:.3f}  y={y:.3f}  z={z:.3f}  site={s.get('site')}")

Found 1 defect site(s)
vacancy_0: x=4.158  y=2.401  z=19.005  site=defect


In [13]:
pos = np.array([s["position"] for s in defect_sites])
print(pos)

[[ 4.15774275  2.40050311 19.00543806]]


In [14]:
from ase.io import read

xyz_path = '/home/shikim/pynta/pynta/preprocessing/custom_surfaces/testing_shinae/defects/pt111_defect_middle.xyz'
geo = read(xyz_path)          # or your current Atoms object
slab = geo.copy()             # if geo already includes adsorbates, slab should be just slab atoms
sites = cas.get_sites()       # or however you build sites
nslab = len(slab)                 # your slab atom count

geoms, site_bond_params_lists, sites_lists = generate_unique_site_additions_vacancy(
    geo=geo,
    sites=sites,
    slab=slab,
    nslab=nslab,
    site_bond_cutoff = 1.5,
    xyz_path=xyz_path,
    site_bond_params_list=[],
    sites_list=[]
    
)
print(len(geoms), "generated geometries")

24 generated geometries


In [15]:
cas = SlabAdsorptionSites(slab, surface=pt111_defect, composition_effect=True)  # ACAT has surface, custom does not find them all!

In [16]:
print(cas.get_unique_sites())
print(len(cas.get_unique_sites()))

[{'site': 'ontop', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([12.47320201,  7.20148072, 19.00418576]), 'normal': array([-3.534e-05,  8.600e-07,  1.000e+00]), 'indices': (27,), 'composition': 'Pt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27])}, {'site': 'bridge', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([ 5.54376719,  7.20142653, 19.00491345]), 'normal': array([-1.3232000e-04,  4.2990000e-05,  9.9999999e-01]), 'indices': (27, 28), 'composition': 'PtPt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28])}, {'site': '3fold', 'surface': 'CustomSurface', 'morphology': 'terrace', 'position': array([ 1.38594787,  0.80021856, 19.0051173 ]), 'normal': array([-5.2499000e-04, -2.5474000e-04,  9.9999983e-01]), 'indices': (27, 28, 30), 'composition': 'PtPtPt', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([27, 28, 30])}]
3


In [17]:
geoms, site_bond_params_lists, sites_lists = generate_unique_site_additions_vacancy(
    geo=geo,
    sites=sites,
    slab=slab,
    nslab=nslab,
    site_bond_cutoff=site_bond_cutoff,
    xyz_path=xyz_path,
)

# ---- print sites_lists ----
print(f"Returned {len(geoms)} geometries")
for i, slist in enumerate(sites_lists):
    print(f"\nGeometry {i}: {len(slist)} site(s)")
    for s in slist:
        pos = np.array(s.get("position", [np.nan, np.nan, np.nan]), dtype=float)
        print(
            f"  name={s.get('name', 'NA'):>10}  "
            f"site={s.get('site', 'NA'):>8}  "
            f"morph={s.get('morphology','NA'):>10}  "
            f"pos=({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})"
        )

# ---- write trajectory of geometries ----
trajfile = "vacancy_unique_sites.traj"
traj = Trajectory(trajfile, "w")
for g in geoms:
    traj.write(g)
traj.close()

print("Wrote trajectory:", trajfile)

Returned 27 geometries

Geometry 0: 1 site(s)
  name=        NA  site=   ontop  morph=   terrace  pos=(12.473, 7.201, 19.004)

Geometry 1: 1 site(s)
  name=        NA  site=  bridge  morph=   terrace  pos=(4.851, 6.001, 19.005)

Geometry 2: 1 site(s)
  name=        NA  site=  bridge  morph=   terrace  pos=(1.386, 0.800, 19.005)

Geometry 3: 1 site(s)
  name=        NA  site=  bridge  morph=   terrace  pos=(5.544, 6.401, 19.005)

Geometry 4: 1 site(s)
  name=        NA  site=  bridge  morph=   terrace  pos=(7.623, 1.200, 19.005)

Geometry 5: 1 site(s)
  name=        NA  site=   3fold  morph=   terrace  pos=(6.930, 0.800, 19.005)

Geometry 6: 1 site(s)
  name=        NA  site=   3fold  morph=   terrace  pos=(11.087, 6.401, 19.005)

Geometry 7: 1 site(s)
  name=        NA  site=   3fold  morph=   terrace  pos=(8.315, 1.600, 19.005)

Geometry 8: 1 site(s)
  name=        NA  site=   ontop  morph=   terrace  pos=(6.930, 7.201, 19.006)

Geometry 9: 1 site(s)
  name=        NA  site=  bridge  

In [18]:
save_sites_to_json(sites_lists, filename="vacancy_sites_lists.json")

Sites data saved to 'vacancy_sites_lists.json' with None replaced by 'null'.
